# 크립토 하이브리드 예측 모델: Chronos + MTGNN (5분봉)

KOSPI `hybrid_model_colab.ipynb`의 크립토 이식 버전.

**입력**: Binance 5분봉 기술지표 20피처 (`data/crypto/features_5m/*.parquet`, 44개 심볼, 로컬 `crypto/preprocess_features.py`가 생성)
**타겟**: `RetTarget_6b` — 30분(6봉) 뒤 수익률 (실거래 루프 리밸런싱 주기와 일치)
**출력**: `crypto_hybrid_best.pt`, `crypto_rl_embeddings.h5` (RL 노트북 입력)

KOSPI 버전과의 차이:
- sequences.h5 사전 생성 없이 노트북 안에서 윈도우 샘플링 (17.7M봉 전체 시퀀스화는 비현실적)
- 정규화: RevIN 사전 처리 대신 윈도우별 z-score
- Chronos 입력: 원시 close 가격 (Chronos 토크나이저가 내부 스케일링)
- 시간 분할: train ~2025-06 / val 2025-07~12 / test 2026-01~07

**사전 준비**: 로컬 `crypto/data/features_5m/` 폴더(2.5GB)를 Drive의
`내 드라이브/졸업프로젝트/data/crypto/features_5m/`에 업로드해둘 것.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q chronos-forecasting einops h5py pyarrow tqdm

In [ ]:
import gc
import glob
import json
import warnings
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from pathlib import Path

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EMB_CACHE  = CRYPTO_DIR / 'crypto_chronos_embs.npz'
RL_EMB_H5  = CRYPTO_DIR / 'crypto_rl_embeddings.h5'
RESULT_JSON= CRYPTO_DIR / 'crypto_hybrid_result.json'

SEQ_LEN    = 60       # 60봉 = 5시간
N_FEATURES = 20
HORIZON    = 6        # 타겟: 6봉(30분) 뒤 수익률
STRIDE     = 6        # 윈도우 샘플링 간격 = 리밸런싱 주기
CHRONOS_DIM= 512      # chronos-t5-small hidden dim

TRAIN_END  = pd.Timestamp('2025-07-01', tz='UTC')   # train: ~2025-06
VAL_END    = pd.Timestamp('2026-01-01', tz='UTC')   # val: 2025-07~12, test: 2026-01~

MAX_TRAIN  = 150_000
MAX_VAL    = 30_000
MAX_TEST   = 30_000

D_MODEL    = 128
DROPOUT    = 0.1
BATCH_SIZE = 256
N_EPOCHS   = 15
LR         = 5e-4
PATIENCE   = 3

FEATURE_COLS = [
    'close', 'open', 'high', 'low', 'volume',
    'SMA_20', 'SMA_60', 'EMA_12', 'EMA_26',
    'MACD', 'MACD_signal', 'MACD_hist', 'RSI_14',
    'BB_upper', 'BB_lower', 'BB_width', 'Volume_ratio',
    'Return_1b', 'Volatility_20b', 'ATR_14',
]  # KOSPI sequences.h5와 동일한 20피처 순서 (close↔Adj_Close, _1b↔_1d, _20b↔_20d)

files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR} — 로컬 features_5m 폴더를 Drive에 업로드했는지 확인'
print(f'심볼 parquet {len(files)}개')

## 1. 데이터 로드 & 윈도우 샘플링

In [ ]:
%%time
# 심볼별 피처 행렬을 메모리에 로드 (~1.5GB) 후,
# STRIDE 간격으로 윈도우 끝 인덱스를 뽑아 시간 기준으로 train/val/test 분할
data = {}
index = {'train': [], 'val': [], 'test': []}

for f in tqdm(files, desc='로드'):
    sym = Path(f).stem
    df = pd.read_parquet(f, columns=['datetime'] + FEATURE_COLS + ['RetTarget_6b'])
    feat = df[FEATURE_COLS].values.astype(np.float32)          # (T, 20)
    data[sym] = {
        'feat': feat,
        'close': feat[:, 0].copy(),                            # 원시 close (Chronos 입력)
        'y': df['RetTarget_6b'].values.astype(np.float32),
        'ts': df['datetime'].values,
    }
    ts = df['datetime']
    for t in range(SEQ_LEN - 1, len(df), STRIDE):              # 윈도우 = 행 [t-59..t], 타겟 = y[t]
        split = 'train' if ts.iloc[t] < TRAIN_END else ('val' if ts.iloc[t] < VAL_END else 'test')
        index[split].append((sym, t))

for k, v in index.items():
    print(f'{k:5s}: 후보 윈도우 {len(v):>9,}개')

In [ ]:
%%time
rng = np.random.default_rng(42)

def build_split(idx_list, cap):
    if len(idx_list) > cap:
        sel = rng.choice(len(idx_list), size=cap, replace=False)
        idx_list = [idx_list[i] for i in np.sort(sel)]
    N = len(idx_list)
    X = np.empty((N, SEQ_LEN, N_FEATURES), np.float32)  # 윈도우별 z-score 정규화
    C = np.empty((N, SEQ_LEN), np.float32)              # 원시 close (Chronos용)
    y = np.empty(N, np.float32)
    for i, (s, t) in enumerate(idx_list):
        w = data[s]['feat'][t - SEQ_LEN + 1 : t + 1]
        mu, sd = w.mean(0), w.std(0) + 1e-8
        X[i] = (w - mu) / sd
        C[i] = data[s]['close'][t - SEQ_LEN + 1 : t + 1]
        y[i] = data[s]['y'][t]
    return X, C, y, idx_list

X_tr, C_tr, y_tr, idx_tr = build_split(index['train'], MAX_TRAIN)
X_va, C_va, y_va, idx_va = build_split(index['val'],   MAX_VAL)
X_te, C_te, y_te, idx_te = build_split(index['test'],  MAX_TEST)

for name, y_ in [('train', y_tr), ('val', y_va), ('test', y_te)]:
    print(f'{name:5s}: {len(y_):>8,}개 | 상승비율 {float((y_ > 0).mean()):.3f} | ret std {float(y_.std()):.5f}')

## 2. Chronos 임베딩 추출 (최초 1회 후 캐시)

In [ ]:
from chronos import ChronosPipeline

gc.collect()
torch.cuda.empty_cache()

pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map=DEVICE,
    torch_dtype=torch.float32,
)
print('Chronos 로드 완료')


@torch.no_grad()
def extract_chronos_embs(close_prices_np, batch_size=256, desc=''):
    """close_prices_np: (N, seq_len) float32 → (N, CHRONOS_DIM) embeddings"""
    tokenizer = pipeline.tokenizer
    encoder   = pipeline.model.model.encoder
    all_embs  = []
    it = range(0, len(close_prices_np), batch_size)
    if desc:
        it = tqdm(it, desc=desc)
    for i in it:
        # tokenizer 내부에서 CPU bucketize 사용 — ctx는 CPU에 두어야 함
        ctx  = torch.tensor(close_prices_np[i:i+batch_size], dtype=torch.float32)
        ids, mask, _ = tokenizer.context_input_transform(ctx)
        out  = encoder(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE))
        h    = out.last_hidden_state                          # (B, L, D)
        m    = mask.to(DEVICE).unsqueeze(-1).float()
        emb  = (h * m).sum(1) / m.sum(1).clamp(min=1)         # masked mean pool
        all_embs.append(emb.cpu().float().numpy())
    return np.concatenate(all_embs, axis=0)

In [ ]:
%%time
if EMB_CACHE.exists():
    cached  = np.load(EMB_CACHE)
    tr_emb, va_emb, te_emb = cached['train'], cached['val'], cached['test']
    assert len(tr_emb) == len(y_tr), '캐시 크기 불일치 — 샘플링 설정이 바뀌었으면 캐시 삭제 후 재추출'
    print(f'캐시 로드: train={tr_emb.shape}, val={va_emb.shape}, test={te_emb.shape}')
else:
    print('Chronos 임베딩 추출 (최초 1회, T4 기준 ~30분)...')
    tr_emb = extract_chronos_embs(C_tr, desc='train')
    va_emb = extract_chronos_embs(C_va, desc='val')
    te_emb = extract_chronos_embs(C_te, desc='test')
    np.savez(EMB_CACHE, train=tr_emb, val=va_emb, test=te_emb)
    print(f'저장 완료: {EMB_CACHE}')

## 3. ChronosMTGNN 모델 정의

KOSPI 버전과 동일한 아키텍처 — Chronos T5 인코더(temporal) + MTGNN 피처 그래프(cross-feature).

In [ ]:
class AdaptiveGraphLearner(nn.Module):
    """학습 가능한 노드 임베딩 → 피처 그래프 인접 행렬 자동 생성"""
    def __init__(self, n_nodes, emb_dim=10):
        super().__init__()
        self.E1 = nn.Parameter(torch.randn(n_nodes, emb_dim) * 0.1)
        self.E2 = nn.Parameter(torch.randn(n_nodes, emb_dim) * 0.1)

    def forward(self):
        return torch.softmax(torch.relu(self.E1 @ self.E2.T), dim=-1)  # (N, N)


class MTGNNEncoder(nn.Module):
    """MTGNN 피처 그래프 인코더 (헤드 없음) → (B, n_features × d_model)"""
    def __init__(self, seq_len=60, n_features=20, d_model=64, n_layers=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Conv2d(1, d_model, kernel_size=(1, 1))

        dilations = [1, 2, 4][:n_layers]
        self.filter_convs = nn.ModuleList([
            nn.Conv2d(d_model, d_model, (1, 3), dilation=(1, d), padding=(0, d))
            for d in dilations
        ])
        self.gate_convs = nn.ModuleList([
            nn.Conv2d(d_model, d_model, (1, 3), dilation=(1, d), padding=(0, d))
            for d in dilations
        ])
        self.res_convs  = nn.ModuleList([nn.Conv2d(d_model, d_model, 1) for _ in dilations])
        self.gc_weights = nn.ModuleList([nn.Linear(d_model, d_model, bias=False) for _ in dilations])
        self.norms      = nn.ModuleList([nn.LayerNorm(d_model) for _ in dilations])
        self.graph = AdaptiveGraphLearner(n_features)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x_flat):
        B, T, N = x_flat.shape
        x = x_flat.permute(0, 2, 1).unsqueeze(1)
        x = self.input_proj(x)                          # (B, d, N, T)
        A = self.graph()                                 # (N, N)

        for fc, gc_c, rc, gc_w, norm in zip(
            self.filter_convs, self.gate_convs, self.res_convs, self.gc_weights, self.norms
        ):
            residual = x
            T_cur    = x.size(-1)
            h = (torch.tanh(fc(x)[..., :T_cur]) *
                 torch.sigmoid(gc_c(x)[..., :T_cur]))   # Gated TCN
            h_p  = h.permute(0, 3, 2, 1)               # (B, T, N, d)
            h_gc = torch.einsum('nm,btmc->btnc', A, h_p)
            h_gc = gc_w(h_gc).permute(0, 3, 2, 1)      # (B, d, N, T)
            combined = self.drop(h_gc) + rc(residual)
            x = norm(combined.permute(0, 3, 2, 1)).permute(0, 3, 2, 1)

        x = x.mean(-1)                      # (B, d, N)
        return x.permute(0, 2, 1).flatten(1)  # (B, N*d)


class ChronosMTGNN(nn.Module):
    """
    Chronos T5 인코더(temporal) + MTGNN 피처 그래프(cross-feature) 하이브리드
    - Chronos: 가격 시계열 시간축 패턴 → 64-dim
    - MTGNN:   20개 기술지표 그래프 관계 → 1280-dim
    → Fusion → 64-dim RL state embedding
    """
    def __init__(self, seq_len=60, n_features=20,
                 chronos_dim=512, d_model=64, n_layers=3, dropout=0.1):
        super().__init__()
        self.chronos_proj = nn.Linear(chronos_dim, d_model)
        self.chronos_norm = nn.LayerNorm(d_model)

        self.mtgnn = MTGNNEncoder(seq_len, n_features, d_model, n_layers, dropout)

        fusion_in = d_model + n_features * d_model   # 64 + 1280 = 1344
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 64),        nn.GELU(),
        )
        self.head = nn.Linear(64, 1)

    def _fuse(self, x_flat, c_emb):
        chron = self.chronos_norm(self.chronos_proj(c_emb))  # (B, 64)
        mtg   = self.mtgnn(x_flat)                           # (B, 1280)
        return self.fusion(torch.cat([chron, mtg], dim=-1))  # (B, 64)

    def forward(self, x_flat, c_emb):
        return self.head(self._fuse(x_flat, c_emb)).squeeze(-1)

    @torch.no_grad()
    def embed(self, x_flat, c_emb):
        """64-dim RL state representation"""
        self.eval()
        return self._fuse(x_flat, c_emb)

    @torch.no_grad()
    def get_adjacency(self):
        return self.mtgnn.graph().cpu().numpy()


model_info = ChronosMTGNN(SEQ_LEN, N_FEATURES, CHRONOS_DIM, d_model=64, n_layers=3)
print(f'ChronosMTGNN 파라미터: {sum(p.numel() for p in model_info.parameters()):,}')
del model_info

## 4. 하이브리드 데이터셋 & 로더

In [ ]:
class HybridDataset(Dataset):
    """메모리 배열(X, chronos emb, y)을 그대로 서빙"""
    def __init__(self, X, embs, y):
        assert len(X) == len(embs) == len(y)
        self.X, self.embs, self.y = X, embs.astype(np.float32), y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return (
            torch.from_numpy(self.X[i]),
            torch.from_numpy(self.embs[i]),
            torch.tensor(self.y[i]),
        )


train_loader = DataLoader(HybridDataset(X_tr, tr_emb, y_tr), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(HybridDataset(X_va, va_emb, y_va), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(HybridDataset(X_te, te_emb, y_te), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Loaders: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)} 배치')

## 5. 학습 & 평가 함수

In [ ]:
def compute_metrics(y_true, y_pred):
    dir_acc  = ((y_true > 0) == (y_pred > 0)).mean()
    spear, _ = spearmanr(y_true, y_pred)
    mae      = np.abs(y_true - y_pred).mean()
    return {
        'dir_acc':  round(float(dir_acc), 4),
        'spearman': round(float(spear), 4),
        'mae':      round(float(mae), 6),
    }


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for x_flat, c_emb, y_ret in loader:
        x_flat, c_emb, y_ret = x_flat.to(DEVICE), c_emb.to(DEVICE), y_ret.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x_flat, c_emb), y_ret)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    for x_flat, c_emb, y_ret in loader:
        pred = model(x_flat.to(DEVICE), c_emb.to(DEVICE)).cpu().numpy()
        preds.append(pred)
        trues.append(y_ret.numpy())
    return compute_metrics(np.concatenate(trues), np.concatenate(preds))


def train_model(model):
    model     = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    criterion = nn.MSELoss()

    best_mae, no_improve, best_state = float('inf'), 0, None
    history = []

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_m   = evaluate(model, val_loader)
        scheduler.step()
        history.append({'epoch': epoch, 'train_loss': tr_loss, **val_m})
        print(f'ep{epoch:02d} | loss={tr_loss:.6f} | val_dir={val_m["dir_acc"]:.4f} | val_mae={val_m["mae"]:.6f}')

        if val_m['mae'] < best_mae:
            best_mae, no_improve = val_m['mae'], 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stop at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    test_m = evaluate(model.to(DEVICE), test_loader)
    print(f'\nTest: {test_m}')
    return model, test_m, history

## 6. 학습 실행

In [ ]:
%%time
gc.collect()
torch.cuda.empty_cache()

hybrid_model, hybrid_metrics, hist = train_model(
    ChronosMTGNN(SEQ_LEN, N_FEATURES, CHRONOS_DIM, d_model=64, n_layers=3, dropout=DROPOUT)
)
torch.save(hybrid_model.state_dict(), MODEL_DIR / 'crypto_hybrid_best.pt')
print('모델 저장 완료: crypto_hybrid_best.pt')

# 결과를 노트북 출력과 별개로 Drive에 즉시 저장
with open(RESULT_JSON, 'w') as f:
    json.dump({'test_metrics': hybrid_metrics, 'history': hist}, f, ensure_ascii=False, indent=2)
print(f'결과 저장 완료: {RESULT_JSON}')

## 7. 결과 평가 & 시각화

In [ ]:
# hybrid_metrics가 커널에 없으면(재시작 등) 저장된 결과 파일에서 복구
if 'hybrid_metrics' not in dir():
    with open(RESULT_JSON) as f:
        _saved = json.load(f)
    hybrid_metrics, hist = _saved['test_metrics'], _saved['history']
    print('저장된 crypto_hybrid_result.json에서 결과 복구')

# KOSPI(일봉) 결과는 스케일이 달라 직접 비교 불가 — 참고용으로만 병기
comparison = {
    'KOSPI Chronos+MTGNN (1일 타겟, 참고)': {'dir_acc': None, 'spearman': None, 'mae': None},  # hybrid_mtgnn_result.json 값으로 채우기
    'Crypto Chronos+MTGNN (30분 타겟)':     hybrid_metrics,
}
results = pd.DataFrame(comparison).T
print(results.to_string())
print(f"\n▶ 랜덤 기준선: dir_acc=0.5, spearman=0.0")

# 학습 곡선
epochs = [h['epoch'] for h in hist]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, [h['train_loss'] for h in hist], marker='o', ms=4)
axes[0].set_title('Train Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[1].plot(epochs, [h['dir_acc'] for h in hist], marker='o', ms=4, color='darkorange')
axes[1].axhline(0.5, color='gray', ls='--', lw=1.2, label='Random')
axes[1].set_title('Val Directional Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.suptitle('Crypto ChronosMTGNN 학습 곡선', fontsize=13)
plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_hybrid_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# MTGNN 학습된 피처 인접 행렬
A = hybrid_model.get_adjacency()
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(A, cmap='YlOrRd', vmin=0, aspect='auto')
ax.set_xticks(range(N_FEATURES))
ax.set_yticks(range(N_FEATURES))
ax.set_xticklabels(FEATURE_COLS, rotation=90, fontsize=8)
ax.set_yticklabels(FEATURE_COLS, fontsize=8)
ax.set_title('Crypto ChronosMTGNN 학습된 피처 인접 행렬', fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_hybrid_adjacency.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. RL용 임베딩 추출

RL 기간(2025-01-01~) 동안 모든 심볼 × 30분 스텝의 64-dim 임베딩을 추출한다.
약 100만 윈도우 — T4 기준 1~2시간 소요. 결과는 `crypto_rl_embeddings.h5`.

In [ ]:
%%time
RL_START = pd.Timestamp('2025-01-01', tz='UTC')   # RL train: 2025년 / RL test: 2026년~
BATCH_W  = 512

hybrid_model.eval()
sym_list, ts_list, emb_list = [], [], []

for sym in tqdm(sorted(data.keys()), desc='심볼'):
    d = data[sym]
    # d['ts']는 셀 4에서 tz-aware Series를 .values로 꺼낼 때 UTC로 정규화된 뒤 tz 정보가
    # 벗겨진 상태 → tz_localize('UTC')로 원래 의미(UTC 시각)를 복원해야 RL_START와 비교 가능
    ts = pd.DatetimeIndex(d['ts']).tz_localize('UTC')
    t_candidates = [t for t in range(SEQ_LEN - 1, len(ts), STRIDE) if ts[t] >= RL_START]
    if not t_candidates:
        continue

    for i in range(0, len(t_candidates), BATCH_W):
        batch_t = t_candidates[i:i+BATCH_W]
        Xb = np.empty((len(batch_t), SEQ_LEN, N_FEATURES), np.float32)
        Cb = np.empty((len(batch_t), SEQ_LEN), np.float32)
        for j, t in enumerate(batch_t):
            w = d['feat'][t - SEQ_LEN + 1 : t + 1]
            mu, sd = w.mean(0), w.std(0) + 1e-8
            Xb[j] = (w - mu) / sd
            Cb[j] = d['close'][t - SEQ_LEN + 1 : t + 1]

        c_emb = torch.tensor(extract_chronos_embs(Cb, batch_size=BATCH_W)).to(DEVICE)
        emb = hybrid_model.embed(torch.tensor(Xb).to(DEVICE), c_emb).cpu().numpy()

        sym_list.extend([sym] * len(batch_t))
        ts_list.extend(ts[batch_t].asi8 // 10**9)   # epoch seconds
        emb_list.append(emb)

embs_arr = np.concatenate(emb_list, axis=0).astype(np.float32)
syms_arr = np.array(sym_list, dtype='S12')
ts_arr   = np.array(ts_list, dtype=np.int64)
print(f'총 {len(embs_arr):,}개 (심볼, 30분 스텝) 임베딩 추출 완료')

In [ ]:
# crypto_rl_embeddings.h5 저장
# 구조: 'symbols'(N, byte str), 'timestamps'(N, epoch sec UTC), 'embeddings'(N, 64)
with h5py.File(RL_EMB_H5, 'w') as f:
    f.create_dataset('symbols',    data=syms_arr, compression='gzip')
    f.create_dataset('timestamps', data=ts_arr,   compression='gzip')
    f.create_dataset('embeddings', data=embs_arr, compression='gzip', chunks=(1000, 64))

print(f'완료: {RL_EMB_H5} ({RL_EMB_H5.stat().st_size/1e6:.1f} MB)')
print()
print('▶ 다음 단계: 크립토 RL 노트북 (PPO 포트폴리오 학습)')